## Surya OCR — тестування

In [1]:
# Підготовка файлів: конвертація/копіювання в docs/png_docs (не в docs_to_md!)
import pillow_avif
from pathlib import Path
from PIL import Image, ImageOps

BASE = Path.cwd().parent
DOCS = BASE / "docs"
OUT_DIR = DOCS / "png_docs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# (підпапка в docs/, ім'я файлу)
FILES = {
    "raport-optimized (шаблон, чистий скан)": ("відпустка", "raport-optimized.avif"),
    "довідка ВЛК (фото паперу)": ("відпустка", "__2023-10-17__121332.png"),
    "public (порожній бланк)": ("відпустка", "public.avif"),
    "1.webp (чистий скан)": ("відпустка", "1.webp"),
    "raport_zvilnennya (рукописний)": ("від руки", "raport_zvilnennya_4037517c79.png"),
}

CMP_IMAGES = {}
for label, (subdir, fname) in FILES.items():
    src_path = DOCS / subdir / fname
    if not src_path.exists():
        print(f"! файл не знайдено, пропускаю: {src_path}")
        continue
    out_path = OUT_DIR / ("cmp_" + Path(fname).stem + ".png")
    # exif_transpose: фото з телефону часто мають поворот у EXIF-метаданих,
    # який Pillow сам по собі ігнорує — без цього зображення може піти в OCR
    # перевернутим, і це ніяк не буде видно з помилки. Це не стосується
    # роздільності/масштабу, тому залишаю.
    img = ImageOps.exif_transpose(Image.open(src_path)).convert("RGB")
    img.save(out_path)
    CMP_IMAGES[label] = str(out_path)

CMP_IMAGES

{'raport-optimized (шаблон, чистий скан)': 'C:\\Users\\Lenovo\\Desktop\\Agentic AI\\project\\docs\\png_docs\\cmp_raport-optimized.png',
 'довідка ВЛК (фото паперу)': 'C:\\Users\\Lenovo\\Desktop\\Agentic AI\\project\\docs\\png_docs\\cmp___2023-10-17__121332.png',
 'public (порожній бланк)': 'C:\\Users\\Lenovo\\Desktop\\Agentic AI\\project\\docs\\png_docs\\cmp_public.png',
 '1.webp (чистий скан)': 'C:\\Users\\Lenovo\\Desktop\\Agentic AI\\project\\docs\\png_docs\\cmp_1.png',
 'raport_zvilnennya (рукописний)': 'C:\\Users\\Lenovo\\Desktop\\Agentic AI\\project\\docs\\png_docs\\cmp_raport_zvilnennya_4037517c79.png'}

### Surya: один спільний сервер інференсу + налаштування під наш випадок

`check_repeatability()` — легка перевірка відтворюваності (n прогонів на тому самому файлі + difflib-схожість між ними, без ШІ-судді й без ручного еталону).

In [2]:
import os
os.environ.setdefault("SURYA_INFERENCE_PARALLEL", "2")

import re, html, difflib
from PIL import Image
from surya.inference import SuryaInferenceManager
from surya.recognition import RecognitionPredictor

manager = SuryaInferenceManager()
recognition_predictor = RecognitionPredictor(manager)

def run_surya(image_path: str) -> str:
    """Розпізнає файл через Surya і повертає чистий текст (без HTML-тегів)."""
    predictions = recognition_predictor([Image.open(image_path)])
    lines = []
    for block in predictions[0].blocks:
        plain = re.sub(r"<br\s*/?>", "\n", block.html)  # <br/> -> новий рядок
        plain = re.sub(r"<[^>]+>", "", plain)            # решту тегів прибрати
        plain = html.unescape(plain).strip()              # &#39; тощо -> звичайні символи
        lines.append(plain)
    return "\n".join(lines)

def check_repeatability(image_path: str, n_runs: int = 2):
    """
    Легка перевірка відтворюваності: прогонити Surya n_runs разів на тому
    самому зображенні й порахувати посимвольну схожість (difflib, без ШІ-судді
    й без ручного еталону) між кожною парою прогонів. Низька схожість на
    "важкому" зображенні — сигнал галюцинації (задокументовано в
    architecture-proposal.md, розділ 8), а не гарантія помилки в конкретному
    прогоні.
    """
    runs = [run_surya(image_path) for _ in range(n_runs)]
    for i, text in enumerate(runs, 1):
        print(f"--- прогін {i}/{n_runs} ---")
        print(text)
        print()
    for i in range(len(runs)):
        for j in range(i + 1, len(runs)):
            ratio = difflib.SequenceMatcher(None, runs[i], runs[j]).ratio()
            print(f"схожість прогін {i + 1} vs {j + 1}: {ratio:.1%}")
    return runs

### Довідка ВЛК (фото паперу)

In [3]:
_ = check_repeatability(CMP_IMAGES["довідка ВЛК (фото паперу)"], n_runs=2)

--- прогін 1/2 ---
ДОВІДКА
військово-лікарської комісії
солдат
(військове звання, прізвання, ім'я та по батькові)
Військова частина: А
в ЗСУ з [REDACTED] квітня 2022 року призваний [REDACTED] РТЦК та СП
(рік народження, військова частина, яким військовим призванням у Збройні Сили, військова професія)
Проведено медичний огляд ВЛК КНП [REDACTED] клінічна лікарня м. Києва. [REDACTED] липня 2023 року.
Діагноз та постанова ВЛК про причинний зв'язок захворювання (травми, поранення, контузії, каліцтва): Стан після мінно – вибухової травми (23.06.2023 р.); закритої черепно-мозкової травми середнього ступеню важкості; струсу головного мозку; вестибулярного синдрому; акубаротравми з пошкодженням обох барабанних перетинок лікованих оперативно: двобічна мірингопластика (28.06.2023р.).
За наказом МОЗ від 04.07.2007 № 370 травма легка.
Травма, ТАК, пов'язана з проходженням військової служби (довідка про обставлення травми не надана).
На підставі статті 81 графи II Розкладу хвороб, потребує відпустки

### raport-optimized (шаблон, чистий скан)

In [4]:
_ = check_repeatability(CMP_IMAGES["raport-optimized (шаблон, чистий скан)"], n_runs=2)

--- прогін 1/2 ---
Командиру військової частини XXX
РАПОРТ
Прошу вас надати мені частину щорічної основної відпустки терміном на 10 (десять) діб із 01.10.2023 року.
Відпустку буду проводити за адресою: країна XXX, місто XXX, вул. XXX (адреса латиницею). Телефон для оповіщення: XXX.
До місця проведення буду добиратися залізничним транспортом за маршрутом: XXX-XXX (10 годин). Автобусним транспортом за маршрутом: XXX (5 годин). Авіасполученням: XXX-XXX (1,5 годин).
До пункту постійної дислокації буду добиратися за маршрутом авіасполученням: XXX-XXX (1,5 годин). Автобусним транспортом за маршрутом: XXX (5 годин). залізничним транспортом за маршрутом: XXX-XXX (10 годин).
*посада/підрозділ* військової частини XXX.
xx.xx.2023p.
ПІБ

--- прогін 2/2 ---
Командиру військової частини XXX
РАПОРТ
Прошу вас надати мені частину щорічної основної відпустки терміном на 10 (десять) діб із 01.10.2023 року.
Відпустку буду проводити за адресою: країна XXX, місто XXX, вул. XXX (адреса латиницею). Телефон дл

### public (порожній бланк)

In [5]:
_ = check_repeatability(CMP_IMAGES["public (порожній бланк)"], n_runs=2)

--- прогін 1/2 ---
Командиру В/ч _____
Рапорт
Прошу надати мені, _____ (звання,
ПГБ), відпустку на _____ діб за сімейними обставинами.
До рапорту додаю:
_____
_____
_____
Надати документи, що підтверджують необхідність надання відпустки за сімейними обставинами
Відпустку булу проводити за адресою:
_____
_____
_____
Тел.: _____
“ _____ ” _____ 2 0 _____ р .
_____
Підпис/ Звання, Прізвище,
Ініціали

--- прогін 2/2 ---
Командиру В/ч _____
Рапорт
Прошу надати мені, _____ (звання,
ПГБ), відпустку на _____ діб за сімейними обставинами.
До рапорту додаю:
_____
_____
_____
Надати документи, що підтверджують необхідність надання відпустки за сімейними обставинами
Відпустку булу проводити за адресою:
_____
_____
_____
Тел.: _____
“ _____ ” _____ 2 0 _____ р .
_____
Підпис/ Звання, Прізвище,
Ініціали

схожість прогін 1 vs 2: 100.0%


### 1.webp (чистий скан)

In [6]:
_ = check_repeatability(CMP_IMAGES["1.webp (чистий скан)"], n_runs=2)

--- прогін 1/2 ---
Начальнику штабу-заступнику
командира військової частини А0000
РАПОРТ
Прошу Вас надати мені частину щорічної основної відпустки за 2023 рік терміном на 10 діб з 01 січні 2023.
Обов'язки помічника начальника штабу прошу покласти на офіцера штабу майора Коцюбу.
Відпустку буду проводити за адресою: м. Вінниця, вул. Велика Бандерівська, б. 11 кв. 1155.
Моб.телефон 067-891-23-45.
У Збройних Силах України з 2014 року.
Помічник начальника штабу
Майор
_____, 2023
В.ПУПКІН
Командиру військової частини А0000
Клопочу по суті рапорту майора Пупкіна.
Начальник штабу-заступник
командира військової частини А0000
генерал
01.01.2023
А.Олексієв

--- прогін 2/2 ---
Начальнику штабу-заступнику
командира військової частини А0000
РАПОРТ
Прошу Вас надати мені частину щорічної основної відпустки за 2023 рік терміном на 10 діб з 01 січні 2023.
Обов'язки помічника начальника штабу прошу покласти на офіцера штабу майора Коцюбу.
Відпустку буду проводити за адресою: м. Вінниця, вул. Велика Банде

### raport_zvilnennya (рукописний)

In [7]:
_ = check_repeatability(CMP_IMAGES["raport_zvilnennya (рукописний)"], n_runs=2)

--- прогін 1/2 ---
NO. ORIGINS
Командиру 3 Запольському
емеративного рауноглумия
провозводнику
Ролерм
Промы Вашого сломатизм перед вынуши люди провозводили
нужда движения шекте бисексовый скульптор и запас проблем
тип промы на простой п.п. "п.п. с 4 ст. 20 Засон". Промы
. Про бисексовый скульптор бесчи 20 в роски.
Зимний и инвалидный перед шекте направим 20 Полемиковского
на бисексовый дом промы шекте направим 20 Полемиковского
На бисексовый дом шекте направим 20 Полемиковского
рабочиков ТКК мо СП шекте Кийв.
По ролерму промы немотрятельно любители солнца провозводить.
1. Копия полерма2. Копия полерма3. Копия связимой про непрерывных4. Копия сормы пломышек позитой5. Копия первых пломышек первомой
1. Копия связимой про шекте по 30
2. Копия непрерывного высылки по 30
3. Копия поверхности зимних инвалид
5. Витек и рестру непрерывные простой
10. Витек и рестру непрерывные простой
Словний ерезейт 2 букву промы спаритивного приможения
неприможения приможения
шекте сражения
04.11.2023
[Handwr